# Neo4j to interactive graph in Google Colab

Connect to a reachable Neo4j database, run a Cypher query, and visualize nodes and relationships using **PyVis**. No changes are made to the database.

**Connection note:** A Neo4j Aura connection usually starts with `neo4j+s://`. A Neo4j instance on `localhost` or a private network cannot be reached directly from Google Colab without network access or a tunnel.

In [ ]:
%pip -q install neo4j pyvis

## 1. Connect (enter credentials at runtime)

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = input('Neo4j URI (e.g. neo4j+s://xxxx.databases.neo4j.io): ').strip()
USER = input('Username [neo4j]: ').strip() or 'neo4j'
PASSWORD = getpass('Password: ')
DATABASE = input('Database [neo4j]: ').strip() or 'neo4j'

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print('Connected successfully!')

## 2. Retrieve a small graph

The query below includes nodes even when they have no outgoing relationship. Change `CYPHER` and `MAX_ROWS` to select a particular subgraph. Keep the result modest for an interactive view.

In [ ]:
MAX_ROWS = 100

CYPHER = '''
MATCH (a)
OPTIONAL MATCH (a)-[r]->(b)
RETURN a, r, b
LIMIT $limit
'''

with driver.session(database=DATABASE) as session:
    records = list(session.run(CYPHER, limit=MAX_ROWS))

print(f'Retrieved {len(records)} rows')

## 3. Draw the graph (drag, zoom, hover)

In [ ]:
from html import escape
from IPython.display import HTML, display
from pyvis.network import Network

net = Network(
    height='650px', width='100%', bgcolor='#ffffff',
    font_color='#1f2937', directed=True, cdn_resources='in_line'
)
seen_nodes = set()
seen_edges = set()

def add_node(node):
    if node is None or node.element_id in seen_nodes:
        return
    labels = sorted(node.labels)
    group = labels[0] if labels else 'Node'
    props = dict(node)
    preferred = ('name', 'title', 'concept_name', 'display_name', 'id', 'code')
    text = next((str(props[k]) for k in preferred if props.get(k) is not None), None)
    text = (text or f'{group} {node.element_id[-6:]}')[:45]
    details = '<br>'.join(
        f'<b>{escape(str(key))}</b>: {escape(str(value)[:200])}'
        for key, value in list(props.items())[:20]
    )
    tooltip = f'<b>{escape(":".join(labels) or "Node")}</b><br>{details}'
    net.add_node(node.element_id, label=text, title=tooltip, group=group, size=20)
    seen_nodes.add(node.element_id)

for row in records:
    a, r, b = row['a'], row['r'], row['b']
    add_node(a)
    add_node(b)
    if r is not None and r.element_id not in seen_edges:
        net.add_edge(
            r.start_node.element_id, r.end_node.element_id,
            label=r.type, title=escape(r.type),
        )
        seen_edges.add(r.element_id)

if not seen_nodes:
    print('No nodes returned. Check the database and Cypher query.')
else:
    net.barnes_hut()
    net.write_html('neo4j_graph.html', notebook=False)
    display(HTML(filename='neo4j_graph.html'))
    print(f'Displayed {len(seen_nodes)} nodes and {len(seen_edges)} relationships.')

## Optional: download the interactive graph

If Colab does not render the graph inline, download the generated HTML file and open it in a browser.

In [ ]:
# Uncomment to download the graph HTML file from Colab:
# from google.colab import files
# files.download('neo4j_graph.html')

**Tip:** To target a specific graph, replace the query with something like `MATCH (a:Person)-[r]->(b) RETURN a,r,b LIMIT $limit`. If you change the return columns, preserve the aliases `a`, `r`, and `b` for the visualization cell.

In [ ]:
# When finished using the connection:
# driver.close()